In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

RAW_DATA_DIR = PROJECT_ROOT / "Data" / "raw"
CLEANED_DATA_DIR = PROJECT_ROOT / "Data" / "cleaned"

CLEANED_DATA_DIR.mkdir(parents=True, exist_ok=True)

PAYMENTS_FILE = RAW_DATA_DIR / "olist_order_payments_dataset.csv"

df = pd.read_csv(PAYMENTS_FILE)

print("Payments dataset loaded successfully.")

Payments dataset loaded successfully.


In [2]:
print("Dataset Shape:", df.shape)

display(df.head())

print("\nData Types:")
display(df.dtypes)

print("\nColumns:")
print(df.columns.tolist())

Dataset Shape: (103886, 5)


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45



Data Types:


order_id                 object
payment_sequential        int64
payment_type             object
payment_installments      int64
payment_value           float64
dtype: object


Columns:
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']


In [3]:
missing_report = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing Percentage": (
        df.isnull().mean() * 100
    ).round(2)
})

display(missing_report)

,Missing Count,Missing Percentage
order_id,0,0.0
payment_sequential,0,0.0
payment_type,0,0.0
payment_installments,0,0.0
payment_value,0,0.0


In [4]:
print("Exact Duplicate Rows:", df.duplicated().sum())

print(
    "Duplicate Payment Records:",
    df.duplicated(
        subset=["order_id", "payment_sequential"]
    ).sum()
)

Exact Duplicate Rows: 0
Duplicate Payment Records: 0


In [5]:
print("Payment Types Distribution:")

display(
    df["payment_type"]
    .value_counts(dropna=False)
    .to_frame("Count")
)

print("\nPayment Types Percentages:")

display(
    (
        df["payment_type"]
        .value_counts(normalize=True, dropna=False) * 100
    ).round(2).to_frame("Percentage")
)

Payment Types Distribution:


,Count
payment_type,
credit_card,76795
boleto,19784
voucher,5775
debit_card,1529
not_defined,3



Payment Types Percentages:


,Percentage
payment_type,
credit_card,73.92
boleto,19.04
voucher,5.56
debit_card,1.47
not_defined,0.00


In [6]:
numerical_columns = [
    "payment_sequential",
    "payment_installments",
    "payment_value"
]

display(df[numerical_columns].describe())

,payment_sequential,payment_installments,payment_value
count,103886.000000,103886.000000,103886.000000
mean,1.092679,2.853349,154.100380
std,0.706584,2.687051,217.494064
min,1.000000,0.000000,0.000000
25%,1.000000,1.000000,56.790000
50%,1.000000,1.000000,100.000000
75%,1.000000,4.000000,171.837500
max,29.000000,24.000000,13664.080000


In [7]:
payment_checks = {
    "Payment Sequential <= 0": (
        df["payment_sequential"] <= 0
    ),
    "Payment Installments <= 0": (
        df["payment_installments"] <= 0
    ),
    "Negative Payment Value": (
        df["payment_value"] < 0
    ),
    "Zero Payment Value": (
        df["payment_value"] == 0
    )
}

for check, condition in payment_checks.items():
    print(f"{check}: {condition.sum()}")

Payment Sequential <= 0: 0
Payment Installments <= 0: 2
Negative Payment Value: 0
Zero Payment Value: 9


In [8]:
payments_per_order = (
    df.groupby("order_id")
    .size()
    .value_counts()
    .sort_index()
)

display(
    payments_per_order.to_frame("Number of Orders")
)

print("Unique Orders:", df["order_id"].nunique())

print("Total Payment Records:", len(df))

,Number of Orders
1,96479
2,2382
3,301
4,108
5,52
6,36
7,28
8,11
9,9
10,5


Unique Orders: 99440
Total Payment Records: 103886


In [9]:
sequence_check = (
    df.groupby("order_id")["payment_sequential"]
    .agg(["count", "nunique", "min", "max"])
)

print(
    "Orders with repeated payment sequence:",
    (df.duplicated(
        subset=["order_id", "payment_sequential"]
    )).sum()
)

display(sequence_check.head(10))

Orders with repeated payment sequence: 0


,count,nunique,min,max
order_id,,,,
00010242fe8c5a6d1ba2dd792cb16214,1,1,1,1
00018f77f2f0320c557190d7a144bdd3,1,1,1,1
000229ec398224ef6ca0657da4fc703e,1,1,1,1
00024acbcdf0a6daa1e931b038114c75,1,1,1,1
00042b26cf59d7ce69dfabb4e55b4fd9,1,1,1,1
00048cc3ae777c65dbb7d2a0634bc1ea,1,1,1,1
00054e8431b9d7675808bcb819fb4a32,1,1,1,1
000576fe39319847cbb9d288c5617fa6,1,1,1,1
0005a1a1728c9d785b8e2b08b904576c,1,1,1,1


In [10]:
# Inspect Suspicious Payment Records

suspicious_payments = df[
    (df["payment_installments"] <= 0) |
    (df["payment_value"] == 0) |
    (df["payment_type"] == "not defined")
].copy()

print("Suspicious Records:", len(suspicious_payments))

display(
    suspicious_payments.sort_values(
        ["payment_type", "payment_value"]
    ).to_string(index=False)
)

Suspicious Records: 11


'                        order_id  payment_sequential payment_type  payment_installments  payment_value\n744bade1fcf9ff3f31d860ace076d422                   2  credit_card                     0          58.69\n1a57108394169c0b47d8f876acc9ba2d                   2  credit_card                     0         129.94\n4637ca194b6387e2d538dc89b124b0ee                   1  not_defined                     1           0.00\n00b1cb0320190ca0daa2c88b35206009                   1  not_defined                     1           0.00\nc8c528189310eaa44a745b8d9d26908b                   1  not_defined                     1           0.00\n8bcbe01d44d147f901cd3192671144db                   4      voucher                     1           0.00\nfa65dad1b0e818e3ccc5cb0e39231352                  14      voucher                     1           0.00\n6ccb433e00daae1283ccc956189c82ae                   4      voucher                     1           0.00\n45ed6e85398a87c253db47c2d9f48216                   3      vouch

In [11]:
# Inspect All Payments for Suspicious Orders

suspicious_order_ids = suspicious_payments["order_id"].unique()

related_payments = df[
    df["order_id"].isin(suspicious_order_ids)
].copy()

related_payments = related_payments.sort_values(
    ["order_id", "payment_sequential"]
)

display(
    related_payments.to_string(index=False)
)

'                        order_id  payment_sequential payment_type  payment_installments  payment_value\n00b1cb0320190ca0daa2c88b35206009                   1  not_defined                     1           0.00\n1a57108394169c0b47d8f876acc9ba2d                   2  credit_card                     0         129.94\n45ed6e85398a87c253db47c2d9f48216                   1      voucher                     1          21.13\n45ed6e85398a87c253db47c2d9f48216                   2      voucher                     1          50.01\n45ed6e85398a87c253db47c2d9f48216                   3      voucher                     1           0.00\n4637ca194b6387e2d538dc89b124b0ee                   1  not_defined                     1           0.00\n6ccb433e00daae1283ccc956189c82ae                   1  credit_card                     5          84.67\n6ccb433e00daae1283ccc956189c82ae                   2      voucher                     1          14.65\n6ccb433e00daae1283ccc956189c82ae                   3      vouch

In [12]:
# Create a working copy

cleaned_df = df.copy()

print("Original Shape:", df.shape)
print("Working Copy Shape:", cleaned_df.shape)

Original Shape: (103886, 5)
Working Copy Shape: (103886, 5)


In [13]:
# Remove leading and trailing spaces

text_columns = cleaned_df.select_dtypes(
    include=["object", "string"]
).columns

for col in text_columns:
    cleaned_df[col] = cleaned_df[col].str.strip()

print("Text columns cleaned.")

Text columns cleaned.


In [14]:
# Standardize payment type values

cleaned_df["payment_type"] = (
    cleaned_df["payment_type"]
    .str.lower()
    .str.strip()
)

print("Payment Types After Standardization:")

display(
    cleaned_df["payment_type"]
    .value_counts(dropna=False)
    .to_frame("Count")
)

Payment Types After Standardization:


,Count
payment_type,
credit_card,76795
boleto,19784
voucher,5775
debit_card,1529
not_defined,3


In [15]:
# Remove exact duplicate rows

duplicates_before = cleaned_df.duplicated().sum()

cleaned_df = cleaned_df.drop_duplicates()

duplicates_after = cleaned_df.duplicated().sum()

print("Duplicates Before:", duplicates_before)
print("Duplicates After:", duplicates_after)
print("Rows Removed:", duplicates_before - duplicates_after)

Duplicates Before: 0
Duplicates After: 0
Rows Removed: 0


In [16]:
# Validate payment identifiers

print("Missing Order IDs:", cleaned_df["order_id"].isna().sum())

print(
    "Duplicate Payment Sequential Records:",
    cleaned_df.duplicated(
        subset=["order_id", "payment_sequential"]
    ).sum()
)

print("Unique Orders:", cleaned_df["order_id"].nunique())

print("Total Payment Records:", len(cleaned_df))

Missing Order IDs: 0
Duplicate Payment Sequential Records: 0
Unique Orders: 99440
Total Payment Records: 103886


In [17]:
# Validate payment types

valid_payment_types = [
    "credit_card",
    "debit_card",
    "boleto",
    "voucher",
    "not_defined"
]

invalid_payment_types = cleaned_df[
    ~cleaned_df["payment_type"].isin(valid_payment_types)
]

print("Invalid Payment Types:", len(invalid_payment_types))

display(invalid_payment_types)

Invalid Payment Types: 0


,order_id,payment_sequential,payment_type,payment_installments,payment_value


In [18]:
# Numerical business rules validation

payment_quality_checks = {
    "Payment Sequential <= 0": (
        cleaned_df["payment_sequential"] <= 0
    ),
    "Payment Installments <= 0": (
        cleaned_df["payment_installments"] <= 0
    ),
    "Negative Payment Value": (
        cleaned_df["payment_value"] < 0
    ),
    "Zero Payment Value": (
        cleaned_df["payment_value"] == 0
    )
}

for check, condition in payment_quality_checks.items():
    print(f"{check}: {condition.sum()}")

Payment Sequential <= 0: 0
Payment Installments <= 0: 2
Negative Payment Value: 0
Zero Payment Value: 9


In [19]:
# Identify suspicious payment records

suspicious_mask = (
    (cleaned_df["payment_installments"] <= 0) |
    (cleaned_df["payment_value"] <= 0) |
    (cleaned_df["payment_type"] == "not_defined") |
    (~cleaned_df["payment_type"].isin(valid_payment_types))
)

suspicious_payments = cleaned_df[
    suspicious_mask
].copy()

print("Suspicious Payment Records:", len(suspicious_payments))

display(suspicious_payments)

Suspicious Payment Records: 11


,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.00
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.00
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.00
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.00
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.00
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.00
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.00
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.00


In [20]:
# Validate payment sequential values

sequence_report = (
    cleaned_df.groupby("order_id")
    .agg(
        payment_records=("payment_sequential", "count"),
        unique_sequences=("payment_sequential", "nunique"),
        min_sequence=("payment_sequential", "min"),
        max_sequence=("payment_sequential", "max")
    )
)

sequence_report["has_sequence_issue"] = (
    (sequence_report["payment_records"] !=
     sequence_report["unique_sequences"]) |
    (sequence_report["min_sequence"] != 1) |
    (
        sequence_report["max_sequence"] !=
        sequence_report["payment_records"]
    )
)

print(
    "Orders with Sequence Issues:",
    sequence_report["has_sequence_issue"].sum()
)

display(
    sequence_report[
        sequence_report["has_sequence_issue"]
    ].head(10)
)

Orders with Sequence Issues: 80


,payment_records,unique_sequences,min_sequence,max_sequence,has_sequence_issue
order_id,,,,,
00ac05fe0fc047c54418098eb64e3aaa,1,1,2,2,True
056c68d093c100017aab1f00f260705c,1,1,2,2,True
0668d086b3ae41adde3aed3dacbc8fae,1,1,2,2,True
0a7d898c6305101e69e9f5a05cd0130d,1,1,2,2,True
0d0e88a418636d40ea780339701db49c,1,1,2,2,True
0fdb6f73fab7005c84d4a43d8a93682b,1,1,2,2,True
159da9914b51de617fe80162939965c5,1,1,2,2,True
166b01d0fce75a595b92ad9e84e0843b,1,1,2,2,True
1896e8b8fd196151559b02e31e98ce89,1,1,2,2,True


In [21]:
# Export payment sequence issues

sequence_issues = (
    sequence_report[
        sequence_report["has_sequence_issue"]
    ]
    .reset_index()
)

sequence_issues.to_csv(
    CLEANED_DATA_DIR / "payments_sequence_issues.csv",
    index=False
)

print("Sequence Issues:", len(sequence_issues))
print("Report saved successfully.")

Sequence Issues: 80
Report saved successfully.


In [22]:
# Final missing values report

missing_report = pd.DataFrame({
    "Missing Count": cleaned_df.isnull().sum(),
    "Missing Percentage": (
        cleaned_df.isnull().mean() * 100
    ).round(2)
})

display(missing_report)

,Missing Count,Missing Percentage
order_id,0,0.0
payment_sequential,0,0.0
payment_type,0,0.0
payment_installments,0,0.0
payment_value,0,0.0


In [23]:
# Build payment quality report

quality_checks = {
    "Total Payment Records": len(cleaned_df),
    "Total Columns": len(cleaned_df.columns),
    "Exact Duplicate Rows": cleaned_df.duplicated().sum(),
    "Missing Order IDs": cleaned_df["order_id"].isna().sum(),
    "Duplicate Payment Sequential Records": (
        cleaned_df.duplicated(
            subset=["order_id", "payment_sequential"]
        ).sum()
    ),
    "Invalid Payment Types": len(invalid_payment_types),
    "Zero Payment Values": (
        cleaned_df["payment_value"] == 0
    ).sum(),
    "Negative Payment Values": (
        cleaned_df["payment_value"] < 0
    ).sum(),
    "Invalid Installments": (
        cleaned_df["payment_installments"] <= 0
    ).sum(),
    "Suspicious Payment Records": len(suspicious_payments),
    "Orders with Sequence Issues": (
        sequence_report["has_sequence_issue"].sum()
    )
}

quality_report = pd.DataFrame(
    list(quality_checks.items()),
    columns=["Quality Check", "Result"]
)

display(quality_report)

,Quality Check,Result
0,Total Payment Records,103886
1,Total Columns,5
2,Exact Duplicate Rows,0
3,Missing Order IDs,0
4,Duplicate Payment Sequential Records,0
5,Invalid Payment Types,0
6,Zero Payment Values,9
7,Negative Payment Values,0
8,Invalid Installments,2
9,Suspicious Payment Records,11


In [24]:
# Final dataset validation

print("Original Records:", len(df))
print("Cleaned Records:", len(cleaned_df))

print("Original Columns:", len(df.columns))
print("Cleaned Columns:", len(cleaned_df.columns))

print("Remaining Exact Duplicates:", cleaned_df.duplicated().sum())

print("Unique Orders:", cleaned_df["order_id"].nunique())

print(
    "Unique Payment Records:",
    cleaned_df.drop_duplicates(
        subset=["order_id", "payment_sequential"]
    ).shape[0]
)

print("\nFinal Data Types:")
display(cleaned_df.dtypes)

Original Records: 103886
Cleaned Records: 103886
Original Columns: 5
Cleaned Columns: 5
Remaining Exact Duplicates: 0
Unique Orders: 99440
Unique Payment Records: 103886

Final Data Types:


order_id                 object
payment_sequential        int64
payment_type             object
payment_installments      int64
payment_value           float64
dtype: object

In [25]:
# Save cleaned dataset and reports

CLEANED_DATA_DIR.mkdir(parents=True, exist_ok=True)

cleaned_file = (
    CLEANED_DATA_DIR / "olist_order_payments_cleaned.csv"
)

quality_file = (
    CLEANED_DATA_DIR / "payments_quality_report.csv"
)

suspicious_file = (
    CLEANED_DATA_DIR / "payments_suspicious_records.csv"
)

sequence_file = (
    CLEANED_DATA_DIR / "payments_sequence_report.csv"
)

cleaned_df.to_csv(cleaned_file, index=False)

quality_report.to_csv(quality_file, index=False)

suspicious_payments.to_csv(suspicious_file, index=False)

sequence_report.to_csv(sequence_file)

print("Files saved successfully:")

print(cleaned_file)
print(quality_file)
print(suspicious_file)
print(sequence_file)

Files saved successfully:
d:\team project\Data\cleaned\olist_order_payments_cleaned.csv
d:\team project\Data\cleaned\payments_quality_report.csv
d:\team project\Data\cleaned\payments_suspicious_records.csv
d:\team project\Data\cleaned\payments_sequence_report.csv


In [26]:
# Verify saved dataset

saved_df = pd.read_csv(cleaned_file)

print("Saved Dataset Shape:", saved_df.shape)
print("Expected Dataset Shape:", cleaned_df.shape)

assert saved_df.shape == cleaned_df.shape

assert (
    saved_df["order_id"].nunique()
    == cleaned_df["order_id"].nunique()
)

assert saved_df.duplicated().sum() == 0

print("Saved dataset verification passed.")

Saved Dataset Shape: (103886, 5)
Expected Dataset Shape: (103886, 5)
Saved dataset verification passed.


In [27]:
# Inspect all orders with payment sequence issues

issue_order_ids = sequence_report[
    sequence_report["has_sequence_issue"]
].index

sequence_issues_details = cleaned_df[
    cleaned_df["order_id"].isin(issue_order_ids)
].sort_values(
    ["order_id", "payment_sequential"]
)

print("Total Affected Orders:", len(issue_order_ids))

print("\nPayment Sequential Distribution:")
display(
    sequence_issues_details["payment_sequential"]
    .value_counts()
    .sort_index()
)

print("\nPayment Records per Affected Order:")
display(
    sequence_issues_details["order_id"]
    .value_counts()
    .value_counts()
    .sort_index()
    .to_frame("Number of Orders")
)

display(sequence_issues_details.head(20))

Total Affected Orders: 80

Payment Sequential Distribution:


payment_sequential
2    80
3     2
Name: count, dtype: int64


Payment Records per Affected Order:


,Number of Orders
count,
1,78
2,2


,order_id,payment_sequential,payment_type,payment_installments,payment_value
27191,00ac05fe0fc047c54418098eb64e3aaa,2,debit_card,1,123.47
12785,056c68d093c100017aab1f00f260705c,2,debit_card,1,52.78
40318,0668d086b3ae41adde3aed3dacbc8fae,2,debit_card,1,252.74
84637,0a7d898c6305101e69e9f5a05cd0130d,2,credit_card,5,215.47
82233,0d0e88a418636d40ea780339701db49c,2,debit_card,1,121.09
62609,0fdb6f73fab7005c84d4a43d8a93682b,2,debit_card,1,34.13
88530,159da9914b51de617fe80162939965c5,2,debit_card,1,40.76
66314,166b01d0fce75a595b92ad9e84e0843b,2,credit_card,6,127.09
87490,1896e8b8fd196151559b02e31e98ce89,2,debit_card,1,108.95
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94
